
# Roxy notebook example: Complexity and low-complexity descriptors

This notebook is a **reference implementation example** for the **complexity and low-complexity descriptor family** in Roxy.

These descriptors try to quantify whether a sequence looks:

- compositionally diverse
- repetitive
- low-complexity
- dominated by a few residues
- locally homogeneous in short windows

They are very useful because two sequences can have similar amino acid composition while having very different **organizational complexity**.

## Covered outputs

This notebook implements examples such as:

- Shannon entropy
- normalized Shannon entropy
- linguistic complexity for k=1,2,3
- unique k-mer fractions
- repeated k-mer fractions
- longest homopolymer run
- homopolymer burden
- low-complexity window burden
- mean and std of window entropy
- compositional inequality (Gini-like proxy)
- dominance of the most frequent residue
- simple class-style implementation for later migration into Roxy

The notebook is designed as a **clean teaching implementation** so it can later become a real `complexity.py` module in Roxy.


In [1]:

from collections import Counter
from itertools import groupby

import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "comp_1",
            "comp_2",
            "comp_3",
            "comp_4",
            "comp_5",
            "comp_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,comp_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,comp_2,GGGGGGGGGGGGGGG,B
2,comp_3,KRRKRRKRRKRRDDDDEE,A
3,comp_4,ACDEFGHIKLMNPQRSTVWY,B
4,comp_5,PPPPGSSSSSTTTTNNQQQ,A
5,comp_6,MSTNPKPQRITLKDGNKVELV,B


## Constants

In [3]:

STANDARD_AA = list("ACDEFGHIKLMNPQRSTVWY")
STANDARD_AA_SET = set(STANDARD_AA)


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA_SET])


def shannon_entropy(seq: str) -> float:
    if len(seq) == 0:
        return np.nan
    counts = Counter(seq)
    probs = np.array([count / len(seq) for count in counts.values()], dtype=float)
    return float(-(probs * np.log2(probs)).sum())


def normalized_shannon_entropy(seq: str) -> float:
    if len(seq) == 0:
        return np.nan
    h = shannon_entropy(seq)
    max_h = np.log2(min(len(seq), len(STANDARD_AA)))
    if max_h == 0:
        return np.nan
    return float(h / max_h)


def linguistic_complexity(seq: str, k: int) -> float:
    if len(seq) < k or k < 1:
        return np.nan
    observed = len({seq[i:i+k] for i in range(len(seq) - k + 1)})
    possible = min(len(seq) - k + 1, 20**k)
    if possible == 0:
        return np.nan
    return observed / possible


def kmers(seq: str, k: int):
    if len(seq) < k:
        return []
    return [seq[i:i+k] for i in range(len(seq) - k + 1)]


def unique_kmer_fraction(seq: str, k: int) -> float:
    words = kmers(seq, k)
    if len(words) == 0:
        return np.nan
    return len(set(words)) / len(words)


def repeated_kmer_fraction(seq: str, k: int) -> float:
    words = kmers(seq, k)
    if len(words) == 0:
        return np.nan
    counts = Counter(words)
    repeated = sum(v for v in counts.values() if v > 1)
    return repeated / len(words)


def longest_homopolymer_run(seq: str) -> int:
    if len(seq) == 0:
        return 0
    return max(len(list(group)) for _, group in groupby(seq))


def homopolymer_burden(seq: str, min_run_length: int = 2) -> float:
    if len(seq) == 0:
        return np.nan
    runs = [len(list(group)) for _, group in groupby(seq)]
    burden = sum(run for run in runs if run >= min_run_length)
    return burden / len(seq)


def windowed_subsequences(seq: str, window: int):
    if len(seq) < window:
        return []
    return [seq[i:i+window] for i in range(len(seq) - window + 1)]


def low_complexity_window_fraction(seq: str, window: int = 5, entropy_threshold: float = 1.5) -> float:
    windows = windowed_subsequences(seq, window)
    if len(windows) == 0:
        return np.nan
    low = sum(shannon_entropy(w) <= entropy_threshold for w in windows)
    return low / len(windows)


def window_entropy_mean(seq: str, window: int = 5) -> float:
    windows = windowed_subsequences(seq, window)
    if len(windows) == 0:
        return np.nan
    values = [shannon_entropy(w) for w in windows]
    return float(np.mean(values))


def window_entropy_std(seq: str, window: int = 5) -> float:
    windows = windowed_subsequences(seq, window)
    if len(windows) == 0:
        return np.nan
    values = [shannon_entropy(w) for w in windows]
    return float(np.std(values, ddof=0))


def most_frequent_residue_fraction(seq: str) -> float:
    if len(seq) == 0:
        return np.nan
    counts = Counter(seq)
    return max(counts.values()) / len(seq)


def gini_like_compositional_inequality(seq: str) -> float:
    if len(seq) == 0:
        return np.nan
    counts = Counter(seq)
    freqs = np.array(sorted([counts.get(aa, 0) / len(seq) for aa in STANDARD_AA]), dtype=float)
    if freqs.sum() == 0:
        return np.nan
    n = len(freqs)
    index = np.arange(1, n + 1)
    gini = (2 * np.sum(index * freqs) / (n * np.sum(freqs))) - (n + 1) / n
    return float(gini)


def residue_dominance_gap(seq: str) -> float:
    if len(seq) == 0:
        return np.nan
    counts = Counter(seq).most_common()
    if len(counts) == 1:
        return 1.0
    return (counts[0][1] - counts[1][1]) / len(seq)


## Core descriptor function

In [5]:

def complexity_descriptors(seq: str, low_complexity_window: int = 5, entropy_threshold: float = 1.5) -> dict:
    seq = clean_sequence(seq)

    out = {
        "comp_length": len(seq),
        "comp_valid_residue_count": len(seq),
        "comp_shannon_entropy": shannon_entropy(seq),
        "comp_shannon_entropy_norm": normalized_shannon_entropy(seq),
        "comp_linguistic_complexity_k1": linguistic_complexity(seq, 1),
        "comp_linguistic_complexity_k2": linguistic_complexity(seq, 2),
        "comp_linguistic_complexity_k3": linguistic_complexity(seq, 3),
        "comp_unique_kmer_fraction_k2": unique_kmer_fraction(seq, 2),
        "comp_unique_kmer_fraction_k3": unique_kmer_fraction(seq, 3),
        "comp_repeated_kmer_fraction_k2": repeated_kmer_fraction(seq, 2),
        "comp_repeated_kmer_fraction_k3": repeated_kmer_fraction(seq, 3),
        "comp_longest_homopolymer_run": longest_homopolymer_run(seq),
        "comp_homopolymer_burden_len2": homopolymer_burden(seq, min_run_length=2),
        "comp_homopolymer_burden_len3": homopolymer_burden(seq, min_run_length=3),
        "comp_low_complexity_window_fraction_w5": low_complexity_window_fraction(
            seq, window=low_complexity_window, entropy_threshold=entropy_threshold
        ),
        "comp_window_entropy_mean_w5": window_entropy_mean(seq, window=low_complexity_window),
        "comp_window_entropy_std_w5": window_entropy_std(seq, window=low_complexity_window),
        "comp_most_frequent_residue_fraction": most_frequent_residue_fraction(seq),
        "comp_gini_like_inequality": gini_like_compositional_inequality(seq),
        "comp_residue_dominance_gap": residue_dominance_gap(seq),
    }

    return out


## Functional usage on one sequence

In [6]:

example = complexity_descriptors(df_demo.loc[0, "sequence"])
list(example.items())[:16]


[('comp_length', 24),
 ('comp_valid_residue_count', 24),
 ('comp_shannon_entropy', 3.438721875540867),
 ('comp_shannon_entropy_norm', 0.7956453231160215),
 ('comp_linguistic_complexity_k1', 0.65),
 ('comp_linguistic_complexity_k2', 0.9565217391304348),
 ('comp_linguistic_complexity_k3', 1.0),
 ('comp_unique_kmer_fraction_k2', 0.9565217391304348),
 ('comp_unique_kmer_fraction_k3', 1.0),
 ('comp_repeated_kmer_fraction_k2', 0.08695652173913043),
 ('comp_repeated_kmer_fraction_k3', 0.0),
 ('comp_longest_homopolymer_run', 2),
 ('comp_homopolymer_burden_len2', 0.25),
 ('comp_homopolymer_burden_len3', 0.0),
 ('comp_low_complexity_window_fraction_w5', 0.15),
 ('comp_window_entropy_mean_w5', 1.9392814698224583)]

## Apply complexity descriptors to the full dataset

In [7]:

df_comp = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(complexity_descriptors).apply(pd.Series),
    ],
    axis=1,
)

df_comp.head()


,sequence_id,sequence,label,comp_length,comp_valid_residue_count,comp_shannon_entropy,comp_shannon_entropy_norm,comp_linguistic_complexity_k1,comp_linguistic_complexity_k2,comp_linguistic_complexity_k3,...,comp_repeated_kmer_fraction_k3,comp_longest_homopolymer_run,comp_homopolymer_burden_len2,comp_homopolymer_burden_len3,comp_low_complexity_window_fraction_w5,comp_window_entropy_mean_w5,comp_window_entropy_std_w5,comp_most_frequent_residue_fraction,comp_gini_like_inequality,comp_residue_dominance_gap
0,comp_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,3.438722,0.795645,0.650000,0.956522,1.000000,...,0.000000,2.0,0.250000,0.000000,0.150000,1.939281,0.394049,0.166667,5.541667e-01,0.000000
1,comp_2,GGGGGGGGGGGGGGG,B,15.0,15.0,-0.000000,-0.000000,0.066667,0.071429,0.076923,...,1.000000,15.0,1.000000,1.000000,1.000000,0.000000,0.000000,1.000000,9.500000e-01,1.000000
2,comp_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,1.836592,0.440438,0.222222,0.411765,0.500000,...,0.750000,4.0,0.777778,0.222222,0.928571,0.949941,0.234133,0.444444,8.500000e-01,0.222222
3,comp_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,4.321928,1.000000,1.000000,1.000000,1.000000,...,0.000000,1.0,0.000000,0.000000,0.000000,2.321928,0.000000,0.050000,-2.220446e-16,0.000000
4,comp_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,2.439268,0.574225,0.315789,0.555556,0.764706,...,0.411765,5.0,0.947368,0.842105,0.800000,0.986742,0.406595,0.263158,7.710526e-01,0.052632


## Inspect complexity descriptor columns

In [8]:

comp_cols = [c for c in df_comp.columns if c.startswith("comp_") and c not in {"comp_length", "comp_valid_residue_count"}]
len(comp_cols), comp_cols[:15]


(18,
 ['comp_shannon_entropy',
  'comp_shannon_entropy_norm',
  'comp_linguistic_complexity_k1',
  'comp_linguistic_complexity_k2',
  'comp_linguistic_complexity_k3',
  'comp_unique_kmer_fraction_k2',
  'comp_unique_kmer_fraction_k3',
  'comp_repeated_kmer_fraction_k2',
  'comp_repeated_kmer_fraction_k3',
  'comp_longest_homopolymer_run',
  'comp_homopolymer_burden_len2',
  'comp_homopolymer_burden_len3',
  'comp_low_complexity_window_fraction_w5',
  'comp_window_entropy_mean_w5',
  'comp_window_entropy_std_w5'])

In [9]:

df_comp[
    [
        "sequence_id",
        "comp_shannon_entropy",
        "comp_shannon_entropy_norm",
        "comp_linguistic_complexity_k2",
        "comp_repeated_kmer_fraction_k2",
        "comp_longest_homopolymer_run",
        "comp_low_complexity_window_fraction_w5",
        "comp_gini_like_inequality",
    ]
]


,sequence_id,comp_shannon_entropy,comp_shannon_entropy_norm,comp_linguistic_complexity_k2,comp_repeated_kmer_fraction_k2,comp_longest_homopolymer_run,comp_low_complexity_window_fraction_w5,comp_gini_like_inequality
0,comp_1,3.438722,0.795645,0.956522,0.086957,2.0,0.150000,5.541667e-01
1,comp_2,-0.000000,-0.000000,0.071429,1.000000,15.0,1.000000,9.500000e-01
2,comp_3,1.836592,0.440438,0.411765,0.823529,4.0,0.928571,8.500000e-01
3,comp_4,4.321928,1.000000,1.000000,0.000000,1.0,0.000000,-2.220446e-16
4,comp_5,2.439268,0.574225,0.555556,0.666667,5.0,0.800000,7.710526e-01
5,comp_6,3.689704,0.853717,1.000000,0.000000,1.0,0.000000,4.452381e-01


## Dataset-level summary

In [10]:

comp_summary = (
    df_comp[comp_cols]
    .mean(axis=0, numeric_only=True)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

comp_summary.head(15)


,descriptor,mean_value
0,comp_longest_homopolymer_run,4.666667
1,comp_shannon_entropy,2.621036
2,comp_window_entropy_mean_w5,1.400362
3,comp_unique_kmer_fraction_k3,0.723605
4,comp_linguistic_complexity_k3,0.723605
5,comp_unique_kmer_fraction_k2,0.665878
6,comp_linguistic_complexity_k2,0.665878
7,comp_shannon_entropy_norm,0.610671
8,comp_gini_like_inequality,0.595076
9,comp_homopolymer_burden_len2,0.495858


## Sanity checks

In [11]:

assert "comp_shannon_entropy" in df_comp.columns
assert "comp_linguistic_complexity_k2" in df_comp.columns
assert "comp_longest_homopolymer_run" in df_comp.columns
assert "comp_low_complexity_window_fraction_w5" in df_comp.columns
assert "comp_gini_like_inequality" in df_comp.columns
assert df_comp["comp_length"].min() > 0

print(f"Number of complexity / low-complexity descriptor columns: {len(comp_cols)}")
print("Complexity descriptor checks passed.")


Number of complexity / low-complexity descriptor columns: 18
Complexity descriptor checks passed.


## Class-style implementation closer to the real package

In [12]:

class ComplexityDescriptors:
    """Example class-style complexity implementation for later migration into Roxy."""

    def __init__(self, low_complexity_window: int = 5, entropy_threshold: float = 1.5):
        self.low_complexity_window = low_complexity_window
        self.entropy_threshold = entropy_threshold

    def transform_sequence(self, seq: str) -> dict:
        return complexity_descriptors(
            seq,
            low_complexity_window=self.low_complexity_window,
            entropy_threshold=self.entropy_threshold,
        )

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


comp_transformer = ComplexityDescriptors(low_complexity_window=5, entropy_threshold=1.5)
comp_matrix = comp_transformer.transform(df_demo["sequence"].tolist())
comp_matrix.head()


,comp_length,comp_valid_residue_count,comp_shannon_entropy,comp_shannon_entropy_norm,comp_linguistic_complexity_k1,comp_linguistic_complexity_k2,comp_linguistic_complexity_k3,comp_unique_kmer_fraction_k2,comp_unique_kmer_fraction_k3,comp_repeated_kmer_fraction_k2,comp_repeated_kmer_fraction_k3,comp_longest_homopolymer_run,comp_homopolymer_burden_len2,comp_homopolymer_burden_len3,comp_low_complexity_window_fraction_w5,comp_window_entropy_mean_w5,comp_window_entropy_std_w5,comp_most_frequent_residue_fraction,comp_gini_like_inequality,comp_residue_dominance_gap
0,24,24,3.438722,0.795645,0.650000,0.956522,1.000000,0.956522,1.000000,0.086957,0.000000,2,0.250000,0.000000,0.150000,1.939281,0.394049,0.166667,5.541667e-01,0.000000
1,15,15,-0.000000,-0.000000,0.066667,0.071429,0.076923,0.071429,0.076923,1.000000,1.000000,15,1.000000,1.000000,1.000000,0.000000,0.000000,1.000000,9.500000e-01,1.000000
2,18,18,1.836592,0.440438,0.222222,0.411765,0.500000,0.411765,0.500000,0.823529,0.750000,4,0.777778,0.222222,0.928571,0.949941,0.234133,0.444444,8.500000e-01,0.222222
3,20,20,4.321928,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,1,0.000000,0.000000,0.000000,2.321928,0.000000,0.050000,-2.220446e-16,0.000000
4,19,19,2.439268,0.574225,0.315789,0.555556,0.764706,0.555556,0.764706,0.666667,0.411765,5,0.947368,0.842105,0.800000,0.986742,0.406595,0.263158,7.710526e-01,0.052632


## Merge transformer output back to the dataset

In [13]:

df_comp_class = pd.concat([df_demo, comp_matrix], axis=1)
df_comp_class.head()


,sequence_id,sequence,label,comp_length,comp_valid_residue_count,comp_shannon_entropy,comp_shannon_entropy_norm,comp_linguistic_complexity_k1,comp_linguistic_complexity_k2,comp_linguistic_complexity_k3,...,comp_repeated_kmer_fraction_k3,comp_longest_homopolymer_run,comp_homopolymer_burden_len2,comp_homopolymer_burden_len3,comp_low_complexity_window_fraction_w5,comp_window_entropy_mean_w5,comp_window_entropy_std_w5,comp_most_frequent_residue_fraction,comp_gini_like_inequality,comp_residue_dominance_gap
0,comp_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,3.438722,0.795645,0.650000,0.956522,1.000000,...,0.000000,2,0.250000,0.000000,0.150000,1.939281,0.394049,0.166667,5.541667e-01,0.000000
1,comp_2,GGGGGGGGGGGGGGG,B,15,15,-0.000000,-0.000000,0.066667,0.071429,0.076923,...,1.000000,15,1.000000,1.000000,1.000000,0.000000,0.000000,1.000000,9.500000e-01,1.000000
2,comp_3,KRRKRRKRRKRRDDDDEE,A,18,18,1.836592,0.440438,0.222222,0.411765,0.500000,...,0.750000,4,0.777778,0.222222,0.928571,0.949941,0.234133,0.444444,8.500000e-01,0.222222
3,comp_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,4.321928,1.000000,1.000000,1.000000,1.000000,...,0.000000,1,0.000000,0.000000,0.000000,2.321928,0.000000,0.050000,-2.220446e-16,0.000000
4,comp_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,2.439268,0.574225,0.315789,0.555556,0.764706,...,0.411765,5,0.947368,0.842105,0.800000,0.986742,0.406595,0.263158,7.710526e-01,0.052632



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move helper logic into `roxy/sequence/complexity.py`
- expose a class such as `ComplexityDescriptors`
- allow configurable:
  - k values for linguistic complexity
  - window size for low-complexity detection
  - entropy threshold
  - which complexity summaries to compute
- add tests for:
  - empty sequences
  - homopolymer-rich sequences
  - highly repetitive k-mer patterns
  - highly diverse sequences
  - lower-case input
  - invalid characters removed during cleaning


## Optional export

In [ ]:
# df_comp.to_csv("demo_complexity_descriptors.csv", index=False)
